#SETUP

In [5]:
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

#EXPERIMENTS

In [4]:
from dotenv import load_dotenv
from pathlib import Path
from langsmith import Client
import os

load_dotenv("C:/Users/Admin/OneDrive/Desktop/MAT496/.env", override=True)

print("PPLX_API_KEY present?", bool(os.environ.get("PPLX_API_KEY")))

client = Client()
local_file_url = "/mnt/data/cd0facce-3b15-462e-9c3b-421a92e08d74.png"
ds = client.create_dataset(dataset_name="My-Examples-From-Notebook")
dataset_id = ds["id"] if isinstance(ds, dict) else ds.id
examples = [
    ( "What does this diagram show?",
      f"The diagram shows a retrieval-augmented generation pipeline. Attachment: {local_file_url}" ),
    ( "What is a RAG pipeline?",
      "RAG fetches external context, then an LLM synthesizes an answer from it." )
]
inputs = [{"question": q, "attachment": local_file_url if "diagram" in q.lower() else None} for q, _ in examples]
outputs = [{"output": a} for _, a in examples]
client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset_id)
print("Uploaded examples to dataset:", dataset_id)


PPLX_API_KEY present? True
Uploaded examples to dataset: 05e2c95c-9c15-4651-8ec1-b70d4c4cf1b0


In [8]:
import os, requests
from langsmith import traceable
PPLX_API_KEY = os.environ.get("PPLX_API_KEY")
if not PPLX_API_KEY:
    raise RuntimeError("PPLX_API_KEY not found")
LOCAL_FILE_URL = "/mnt/data/cd0facce-3b15-462e-9c3b-421a92e08d74.png"
MODEL = "sonar-pro"
class PerplexityClient:
    def __init__(self, key):
        self.key = key
        self.base = "https://api.perplexity.ai"
        self.headers = {"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"}
    def generate(self, messages, model=MODEL, temperature=0.0, max_tokens=600):
        payload = {"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens}
        r = requests.post(f"{self.base}/chat/completions", headers=self.headers, json=payload, timeout=60)
        r.raise_for_status()
        data = r.json()
        if isinstance(data.get("choices"), list) and data["choices"]:
            ch = data["choices"][0]
            if isinstance(ch.get("message"), dict):
                return ch["message"].get("content","")
            return ch.get("text","")
        return str(data)
pplx = PerplexityClient(PPLX_API_KEY)
@traceable(run_type="chain")
def simple_retrieve(q: str):
    return [f"Simulated doc: summary about {q}", f"Attachment: {LOCAL_FILE_URL}"]
@traceable(run_type="chain")
def generate_response(question: str, documents):
    ctx = "\n\n".join(documents)
    system = "You are a concise assistant. Use the context and attachments to answer in <=3 sentences."
    user = f"Context:\n{ctx}\n\nQuestion: {question}"
    messages = [{"role":"system","content":system},{"role":"user","content":user}]
    return call_perplexity(messages)
@traceable(run_type="llm", metadata={"ls_provider":"perplexity","ls_model_name":MODEL})
def call_perplexity(messages):
    return pplx.generate(messages)
@traceable(run_type="chain")
def langsmith_rag(question: str):
    docs = simple_retrieve(question)
    out = generate_response(question, docs)
    return out


In [9]:
from dotenv import load_dotenv
from langsmith import Client
load_dotenv()
client = Client()
dataset_name = "Perplexity-RAG-Golden"
ds = client.create_dataset(dataset_name=dataset_name)
dataset_id = ds["id"] if isinstance(ds, dict) else ds.id
examples = [
    ("What is Perplexity?", "Perplexity is an AI search and answer engine that fuses retrieval with LLM reasoning."),
    ("How to call Perplexity API?", "Authenticate with a bearer token using the PPLX_API_KEY header and call /chat/completions."),
    ("What does the attached diagram show?", f"The attached file shows a RAG pipeline diagram. Attachment: {LOCAL_FILE_URL}")
]
inputs = [{"question": q, "attachment": (LOCAL_FILE_URL if "attached" in q.lower() or "attached" in a.lower() else None)} for q,a in examples]
outputs = [{"output": a} for q,a in examples]
client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset_id)
print("dataset_id", dataset_id)


dataset_id c69c9b35-5fd3-4940-9401-9dc14569ec34


In [12]:
from langsmith import evaluate
def is_precise(reference_outputs: dict, outputs: dict) -> dict:
    ref = reference_outputs.get("output","")
    out = outputs.get("output","")
    score = int(len(out) <= 1.2 * len(ref))
    return {"key":"is_precise","score":score}
def target_function(inputs: dict):
    return langsmith_rag(inputs["question"])
evaluate(target_function, data=dataset_id, evaluators=[is_precise], experiment_prefix="sonar-pro")


View the evaluation results for experiment: 'sonar-pro-4aae46b4' at:
https://smith.langchain.com/o/1861e497-c836-4ce2-af46-5293bd7cc8c4/datasets/c69c9b35-5fd3-4940-9401-9dc14569ec34/compare?selectedSessions=760fc690-266b-4b36-a9f1-32a69e63c933




0it [00:00, ?it/s]

,inputs.question,inputs.attachment,outputs.output,error,reference.output,feedback.is_precise,execution_time,example_id,id
0,What is Perplexity?,None,**Perplexity** is an AI-powered answer engine ...,None,Perplexity is an AI search and answer engine t...,0,4.748341,875c9f88-5c74-45ce-8b38-835611d8489c,019ab47d-9be6-7032-83ad-5775b5b08f6d
1,What does the attached diagram show?,/mnt/data/cd0facce-3b15-462e-9c3b-421a92e08d74...,"The attached diagram is a **context diagram**,...",None,The attached file shows a RAG pipeline diagram...,0,3.651106,b459d0b9-ec4e-47bc-a1b1-7bb393d3a02c,019ab47d-ae79-7510-ab14-5c7c3f1c51b1
2,How to call Perplexity API?,None,"To call the **Perplexity API**, first generate...",None,Authenticate with a bearer token using the PPL...,0,4.759038,dfd6d920-7839-4cc6-a533-553d8a64cf18,019ab47d-bcc0-759a-ae55-a8fe2242b9da
